In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import pickle
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import yaml
import matplotlib.dates as mdates
import re

from neuralhydrology.modelzoo import get_model
from neuralhydrology.utils.config import Config
from neuralhydrology.modelzoo.cudalstm import CudaLSTM
from neuralhydrology.modelzoo.customlstm import CustomLSTM

In [2]:
# ------- Paths -------
model_dir=Path('../final_runs/runs/precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2302_000856')
pickle_file_dir = model_dir / 'validation' / 'model_epoch030' / 'validation_results.p'
pt_file_dir = model_dir / 'model_epoch030.pt'
optimizer_file_dir = model_dir / 'optimizer_state_epoch030.pt'
scaler_path = model_dir / "train_data" / "train_data_scaler.yml"
esp_hydrographs_output_dir = Path("./hydrographs_esp/ensemble_precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2302_000856")
attributes_file = Path("../final_runs/data/attributes/attributes.csv")
ts_file = Path("../final_runs/data/time_series")

In [3]:
runs_root = Path("../final_runs/runs")
model_pattern = "precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_*"
model_dirs = sorted([p for p in runs_root.glob(model_pattern) if p.is_dir()])

print(f"Found {len(model_dirs)} model folders")
for p in model_dirs:
    m = re.search(r"seed_(\d+)_", p.name)
    print(f"seed={m.group(1) if m else 'NA'} -> {p.name}")

# Optional: combined output dir (instead of one seed-specific folder)
esp_hydrographs_output_dir = Path("./hydrographs_esp/multi_seed_ensemble")
esp_hydrographs_output_dir.mkdir(parents=True, exist_ok=True)

# ...existing code...

Found 8 model folders
seed=111 -> precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_111_2302_000856
seed=222 -> precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_222_2302_001500
seed=333 -> precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_333_2302_002104
seed=444 -> precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_444_2302_002709
seed=555 -> precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_555_2302_121416
seed=666 -> precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_666_2302_122033
seed=777 -> precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_777_2302_122647
seed=888 -> precip_prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day_prcp_gauge_mm_day_seed_888_2302_123303


In [4]:
pt_data = torch.load(pt_file_dir,weights_only=True)
# pt_data

In [5]:
cfg = Config(model_dir / "config.yml")

In [6]:
optimized_model = CudaLSTM(cfg)
# print(optimized_model)

state_dict = torch.load(pt_file_dir)

optimized_model.load_state_dict(state_dict)
optimized_model.eval()

/tmp/ipykernel_24741/2693144925.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pt_file_dir)


CudaLSTM(
  (embedding_net): InputLayer(
    (statics_embedding): Identity()
    (dynamics_embeddings): ModuleList(
      (0): Identity()
    )
  )
  (lstm): LSTM(18, 256)
  (dropout): Dropout(p=0.4, inplace=False)
  (head): Regression(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=1, bias=True)
    )
  )
)

In [7]:
# SCALER
with open(scaler_path, "r") as f:
    scaler = yaml.safe_load(f)

In [8]:
# DYNAMIC INPUTS — now takes scaler as argument
def norm_dyn(tensor, varname, scaler):
    center = scaler["xarray_feature_center"]["data_vars"][varname]["data"]
    scale  = scaler["xarray_feature_scale"]["data_vars"][varname]["data"]
    center = torch.tensor(center, dtype=tensor.dtype, device=tensor.device)
    scale  = torch.tensor(scale,  dtype=tensor.dtype, device=tensor.device)
    return (tensor - center) / scale

In [9]:
basins= ["CAMELS_UY_2",
    "CAMELS_UY_3",
    "CAMELS_UY_5",
    "CAMELS_UY_6",
    "CAMELS_UY_7",
    "CAMELS_UY_8",
    "CAMELS_UY_9",
    "CAMELS_UY_10",
    "CAMELS_UY_11",
    "CAMELS_UY_15",
    "CAMELS_UY_16"]

## Plotting ESP against observed values

In [32]:
# # Load attributes once (outside all loops)
# df_attr = pd.read_csv(attributes_file, index_col=0)
# df_attr.index = df_attr.index.astype(str)

# static_features = [
#     "elev_mean", "slope_mean", "area_gages2",
#     "sand_frac", "silt_frac", "clay_frac",
#     "p_mean", "pet_mean", "aridity",
#     "high_prec_dur", "low_prec_dur"
# ]

# dynamic_vars = ["QObs_mm_d","srad_W_m2","tmax_C","tmin_C",
#                 "prcp_mm_day","prcp_chirps_mm_day","prcp_mswep_mm_day","prcp_gauge_mm_day"]

# for basin in basins:
#     # Load timeseries once per basin
#     ts_file_dir = ts_file / f"{basin}.nc"
#     ds = xr.open_dataset(ts_file_dir)
#     df_dyn = ds[dynamic_vars].to_dataframe().sort_index()

#     for month in range(1, 13):
#         # ---- Collect predictions from ALL seeds, keyed by year ----
#         # year_preds[year] = list of arrays, one per seed
#         year_preds = {yr: [] for yr in range(1989, 2008)}

#         for model_dir in model_dirs:
#             pt_file_dir = model_dir / 'model_epoch030.pt'
#             scaler_path = model_dir / "train_data" / "train_data_scaler.yml"
#             cfg = Config(model_dir / "config.yml")

#             # Load THIS model's weights
#             cur_model = CudaLSTM(cfg)
#             cur_model.load_state_dict(torch.load(pt_file_dir, weights_only=True))
#             cur_model.eval()

#             custom_lstm = CustomLSTM(cfg=cfg)
#             custom_lstm.copy_weights(cur_model)
#             custom_lstm.eval()

#             with open(scaler_path, "r") as f:
#                 scaler = yaml.safe_load(f)

#             # Normalize statics with THIS model's scaler
#             static_values = df_attr.loc[basin, static_features].to_numpy(dtype=np.float32)
#             attr_means = np.array([scaler["attribute_means"][k] for k in static_features], dtype=np.float32)
#             attr_stds  = np.array([scaler["attribute_stds"][k]  for k in static_features], dtype=np.float32)
#             static_norm = (static_values - attr_means) / attr_stds
#             static_tensor = torch.tensor(static_norm).unsqueeze(0)

#             # Spinup
#             spinup_start = pd.to_datetime("1989-10-01")
#             prediction_start = pd.Timestamp(year=2008, month=month, day=1)
#             spinup_end = prediction_start - pd.Timedelta(days=1)

#             spinup_df = df_dyn.loc[spinup_start:spinup_end][dynamic_vars]
#             spinup_tensor = torch.tensor(spinup_df.values.astype(np.float32)).unsqueeze(0)

#             x_srad    = spinup_tensor[..., 1:2]
#             x_tmax    = spinup_tensor[..., 2:3]
#             x_tmin    = spinup_tensor[..., 3:4]
#             x_prcp    = spinup_tensor[..., 4:5]
#             x_prcp_ch = spinup_tensor[..., 5:6]
#             x_prcp_ms = spinup_tensor[..., 6:7]
#             x_prcp_ga = spinup_tensor[..., 7:8]

#             spinup_inputs = {
#                 "x_d": {
#                     "srad_W_m2":          norm_dyn(x_srad,    "srad_W_m2", scaler),
#                     "tmax_C":             norm_dyn(x_tmax,    "tmax_C", scaler),
#                     "tmin_C":             norm_dyn(x_tmin,    "tmin_C", scaler),
#                     "prcp_mm_day":        norm_dyn(x_prcp,    "prcp_mm_day", scaler),
#                     "prcp_chirps_mm_day": norm_dyn(x_prcp_ch, "prcp_chirps_mm_day", scaler),
#                     "prcp_mswep_mm_day":  norm_dyn(x_prcp_ms, "prcp_mswep_mm_day", scaler),
#                     "prcp_gauge_mm_day":  norm_dyn(x_prcp_ga, "prcp_gauge_mm_day", scaler),
#                 },
#                 "x_s": static_tensor
#             }

#             with torch.no_grad():
#                 out = custom_lstm(spinup_inputs)
#             h_complete_run = out["h_n"][:, -1, :]
#             c_complete_run = out["c_n"][:, -1, :]

#             # ESP members for this seed
#             for year in range(1989, 2008):
#                 start = pd.Timestamp(year=year, month=month, day=1)
#                 end = start + pd.DateOffset(months=3) - pd.Timedelta(days=1)

#                 historical_df = df_dyn.loc[start:end][dynamic_vars].copy()
#                 historical_df.index = historical_df.index.map(lambda d: d.replace(year=2008))
#                 historical_tensor = torch.tensor(historical_df.values.astype(np.float32)).unsqueeze(0)

#                 x_srad_h    = historical_tensor[..., 1:2]
#                 x_tmax_h    = historical_tensor[..., 2:3]
#                 x_tmin_h    = historical_tensor[..., 3:4]
#                 x_prcp_h    = historical_tensor[..., 4:5]
#                 x_prcp_ch_h = historical_tensor[..., 5:6]
#                 x_prcp_ms_h = historical_tensor[..., 6:7]
#                 x_prcp_ga_h = historical_tensor[..., 7:8]

#                 historical_inputs = {
#                     "x_d": {
#                         "srad_W_m2":          norm_dyn(x_srad_h,    "srad_W_m2", scaler),
#                         "tmax_C":             norm_dyn(x_tmax_h,    "tmax_C", scaler),
#                         "tmin_C":             norm_dyn(x_tmin_h,    "tmin_C", scaler),
#                         "prcp_mm_day":        norm_dyn(x_prcp_h,    "prcp_mm_day", scaler),
#                         "prcp_chirps_mm_day": norm_dyn(x_prcp_ch_h, "prcp_chirps_mm_day", scaler),
#                         "prcp_mswep_mm_day":  norm_dyn(x_prcp_ms_h, "prcp_mswep_mm_day", scaler),
#                         "prcp_gauge_mm_day":  norm_dyn(x_prcp_ga_h, "prcp_gauge_mm_day", scaler),
#                     },
#                     "x_s": static_tensor
#                 }

#                 with torch.no_grad():
#                     out = custom_lstm(historical_inputs, h_0=h_complete_run, c_0=c_complete_run)

#                 y_hat = out["y_hat"]
#                 q_center = torch.tensor(scaler["xarray_feature_center"]["data_vars"]["QObs_mm_d"]["data"], dtype=y_hat.dtype)
#                 q_scale  = torch.tensor(scaler["xarray_feature_scale"]["data_vars"]["QObs_mm_d"]["data"], dtype=y_hat.dtype)
#                 y_hat_denorm = y_hat * q_scale + q_center

#                 year_preds[year].append(y_hat_denorm[0, :, 0].cpu().numpy())

#         # ---- Average across seeds per year, then stack into ensemble ----
#         ensemble_preds = []
#         for year in range(1989, 2008):
#             seed_arrays = year_preds[year]  # list of 8 arrays
#             min_seed_len = min(len(a) for a in seed_arrays)
#             stacked = np.vstack([a[:min_seed_len] for a in seed_arrays])  # [n_seeds, n_days]
#             mean_across_seeds = np.nanmean(stacked, axis=0)  # [n_days]
#             ensemble_preds.append(mean_across_seeds)

#         # ---- Observations (only need once per basin/month) ----
#         obs_3_months = []
#         for year in range(1989, 2008):
#             start = pd.Timestamp(year=year, month=month, day=1)
#             end = start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
#             obs_vals = df_dyn.loc[f"{start}":f"{end}", 'QObs_mm_d'].values
#             obs_3_months.append(obs_vals)

#         # ---- Align & plot ----
#         ref_start = pd.Timestamp(year=2008, month=month, day=1)
#         ref_end = ref_start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
#         dates = pd.date_range(start=ref_start, end=ref_end)

#         min_len = min(
#             min(len(p) for p in ensemble_preds),
#             min(len(o) for o in obs_3_months)
#         )
#         dates = dates[:min_len]

#         all_preds = np.vstack([p[:min_len] for p in ensemble_preds])
#         median_pred = np.nanmedian(all_preds, axis=0)
#         percentile_10 = np.nanpercentile(all_preds, 10, axis=0)
#         percentile_90 = np.nanpercentile(all_preds, 90, axis=0)

#         all_obs = np.vstack([o[:min_len] for o in obs_3_months])
#         median_obs = np.nanmedian(all_obs, axis=0)
#         percentile_10_obs = np.nanpercentile(all_obs, 10, axis=0)
#         percentile_90_obs = np.nanpercentile(all_obs, 90, axis=0)
#         percentile_0_obs = np.nanpercentile(all_obs, 0, axis=0)
#         percentile_100_obs = np.nanpercentile(all_obs, 100, axis=0)

#         fig, ax = plt.subplots(figsize=(12, 5))
#         ax.fill_between(dates, percentile_10_obs, percentile_90_obs, color="blue", alpha=0.2, label="10th-90th Percentile (Obs)")
#         ax.fill_between(dates, percentile_0_obs, percentile_100_obs, color="blue", alpha=0.1, label="Full distribution (Obs)")
#         ax.fill_between(dates, percentile_10, percentile_90, color="red", alpha=0.2, label="10th-90th Percentile (Pred)")
#         ax.plot(dates, median_obs, color="blue", label="Median Observed (Obs)", linewidth=2)
#         ax.plot(dates, median_pred, color="red", label="Median Prediction (Pred)", linewidth=2)
#         ax.set_xlabel("Date")
#         ax.set_ylabel("Streamflow (mm/d)")
#         ax.set_title(f"Ensemble streamflow prediction (multi-seed) - Basin: {basin} - Month: {month}")
#         ax.legend()
#         ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
#         fig.savefig(esp_hydrographs_output_dir / f"{basin}_month_{month}_hydrograph.png", dpi=300)
#         plt.close(fig)

#         print(f"Plotted month {month} for basin {basin}")

## Plotting ESP against simulated values

In [31]:
# Load attributes once (outside all loops)
df_attr = pd.read_csv(attributes_file, index_col=0)
df_attr.index = df_attr.index.astype(str)

static_features = [
    "elev_mean", "slope_mean", "area_gages2",
    "sand_frac", "silt_frac", "clay_frac",
    "p_mean", "pet_mean", "aridity",
    "high_prec_dur", "low_prec_dur"
]

dynamic_vars = ["QObs_mm_d","srad_W_m2","tmax_C","tmin_C",
                "prcp_mm_day","prcp_chirps_mm_day","prcp_mswep_mm_day","prcp_gauge_mm_day"]

for basin in basins:
    # Load timeseries once per basin
    ts_file_dir = ts_file / f"{basin}.nc"
    ds = xr.open_dataset(ts_file_dir)
    df_dyn = ds[dynamic_vars].to_dataframe().sort_index()

    for month in range(1, 13):
        # ---- Collect predictions from ALL seeds, keyed by year ----
        # year_preds[year] = list of arrays, one per seed
        year_preds = {yr: [] for yr in range(1989, 2008)}
        # year_sim[year] = list of arrays (simulated spinup Q), one per seed
        year_sim = {yr: [] for yr in range(1989, 2008)}

        for model_dir in model_dirs:
            pt_file_dir = model_dir / 'model_epoch030.pt'
            scaler_path = model_dir / "train_data" / "train_data_scaler.yml"
            cfg = Config(model_dir / "config.yml")

            # Load THIS model's weights
            cur_model = CudaLSTM(cfg)
            cur_model.load_state_dict(torch.load(pt_file_dir, weights_only=True))
            cur_model.eval()

            custom_lstm = CustomLSTM(cfg=cfg)
            custom_lstm.copy_weights(cur_model)
            custom_lstm.eval()

            with open(scaler_path, "r") as f:
                scaler = yaml.safe_load(f)

            # Normalize statics with THIS model's scaler
            static_values = df_attr.loc[basin, static_features].to_numpy(dtype=np.float32)
            attr_means = np.array([scaler["attribute_means"][k] for k in static_features], dtype=np.float32)
            attr_stds  = np.array([scaler["attribute_stds"][k]  for k in static_features], dtype=np.float32)
            static_norm = (static_values - attr_means) / attr_stds
            static_tensor = torch.tensor(static_norm).unsqueeze(0)

            # Spinup
            spinup_start = pd.to_datetime("1989-10-01")
            prediction_start = pd.Timestamp(year=2008, month=month, day=1)
            spinup_end = prediction_start - pd.Timedelta(days=1)

            spinup_df = df_dyn.loc[spinup_start:spinup_end][dynamic_vars]
            spinup_tensor = torch.tensor(spinup_df.values.astype(np.float32)).unsqueeze(0)

            x_srad    = spinup_tensor[..., 1:2]
            x_tmax    = spinup_tensor[..., 2:3]
            x_tmin    = spinup_tensor[..., 3:4]
            x_prcp    = spinup_tensor[..., 4:5]
            x_prcp_ch = spinup_tensor[..., 5:6]
            x_prcp_ms = spinup_tensor[..., 6:7]
            x_prcp_ga = spinup_tensor[..., 7:8]

            spinup_inputs = {
                "x_d": {
                    "srad_W_m2":          norm_dyn(x_srad,    "srad_W_m2", scaler),
                    "tmax_C":             norm_dyn(x_tmax,    "tmax_C", scaler),
                    "tmin_C":             norm_dyn(x_tmin,    "tmin_C", scaler),
                    "prcp_mm_day":        norm_dyn(x_prcp,    "prcp_mm_day", scaler),
                    "prcp_chirps_mm_day": norm_dyn(x_prcp_ch, "prcp_chirps_mm_day", scaler),
                    "prcp_mswep_mm_day":  norm_dyn(x_prcp_ms, "prcp_mswep_mm_day", scaler),
                    "prcp_gauge_mm_day":  norm_dyn(x_prcp_ga, "prcp_gauge_mm_day", scaler),
                },
                "x_s": static_tensor
            }

            with torch.no_grad():
                out = custom_lstm(spinup_inputs)
            h_complete_run = out["h_n"][:, -1, :]
            c_complete_run = out["c_n"][:, -1, :]
            y_spinup = out["y_hat"]  # [1, n_spinup_days, 1]

            # Denormalize y_spinup
            q_center = torch.tensor(scaler["xarray_feature_center"]["data_vars"]["QObs_mm_d"]["data"], dtype=y_spinup.dtype)
            q_scale  = torch.tensor(scaler["xarray_feature_scale"]["data_vars"]["QObs_mm_d"]["data"], dtype=y_spinup.dtype)
            y_spinup_denorm = (y_spinup * q_scale + q_center)[0, :, 0].cpu().numpy()

            # Create date-indexed series for simulated spinup Q
            spinup_sim_series = pd.Series(y_spinup_denorm, index=spinup_df.index)

            # Extract simulated Q for each year's 3-month window from spinup
            for year in range(1989, 2008):
                start = pd.Timestamp(year=year, month=month, day=1)
                end = start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
                sim_vals = spinup_sim_series.loc[start:end].values
                if len(sim_vals) > 0:
                    year_sim[year].append(sim_vals)

            # ESP members for this seed
            for year in range(1989, 2008):
                start = pd.Timestamp(year=year, month=month, day=1)
                end = start + pd.DateOffset(months=3) - pd.Timedelta(days=1)

                historical_df = df_dyn.loc[start:end][dynamic_vars].copy()
                historical_df.index = historical_df.index.map(lambda d: d.replace(year=2008))
                historical_tensor = torch.tensor(historical_df.values.astype(np.float32)).unsqueeze(0)

                x_srad_h    = historical_tensor[..., 1:2]
                x_tmax_h    = historical_tensor[..., 2:3]
                x_tmin_h    = historical_tensor[..., 3:4]
                x_prcp_h    = historical_tensor[..., 4:5]
                x_prcp_ch_h = historical_tensor[..., 5:6]
                x_prcp_ms_h = historical_tensor[..., 6:7]
                x_prcp_ga_h = historical_tensor[..., 7:8]

                historical_inputs = {
                    "x_d": {
                        "srad_W_m2":          norm_dyn(x_srad_h,    "srad_W_m2", scaler),
                        "tmax_C":             norm_dyn(x_tmax_h,    "tmax_C", scaler),
                        "tmin_C":             norm_dyn(x_tmin_h,    "tmin_C", scaler),
                        "prcp_mm_day":        norm_dyn(x_prcp_h,    "prcp_mm_day", scaler),
                        "prcp_chirps_mm_day": norm_dyn(x_prcp_ch_h, "prcp_chirps_mm_day", scaler),
                        "prcp_mswep_mm_day":  norm_dyn(x_prcp_ms_h, "prcp_mswep_mm_day", scaler),
                        "prcp_gauge_mm_day":  norm_dyn(x_prcp_ga_h, "prcp_gauge_mm_day", scaler),
                    },
                    "x_s": static_tensor
                }

                with torch.no_grad():
                    out = custom_lstm(historical_inputs, h_0=h_complete_run, c_0=c_complete_run)

                y_hat = out["y_hat"]
                y_hat_denorm = y_hat * q_scale + q_center

                year_preds[year].append(y_hat_denorm[0, :, 0].cpu().numpy())

        # ---- Average across seeds per year, then stack into ensemble ----
        ensemble_preds = []
        for year in range(1989, 2008):
            seed_arrays = year_preds[year]  # list of arrays, one per seed
            min_seed_len = min(len(a) for a in seed_arrays)
            stacked = np.vstack([a[:min_seed_len] for a in seed_arrays])  # [n_seeds, n_days]
            mean_across_seeds = np.nanmean(stacked, axis=0)  # [n_days]
            ensemble_preds.append(mean_across_seeds)

        # ---- Simulated spinup Q (averaged across seeds per year) ----
        sim_3_months = []
        for year in range(1989, 2008):
            if year_sim[year]:  # skip years with no spinup coverage
                seed_arrays = year_sim[year]
                min_seed_len = min(len(a) for a in seed_arrays)
                stacked = np.vstack([a[:min_seed_len] for a in seed_arrays])
                mean_across_seeds = np.nanmean(stacked, axis=0)
                sim_3_months.append(mean_across_seeds)

        # ---- Align & plot ----
        ref_start = pd.Timestamp(year=2008, month=month, day=1)
        ref_end = ref_start + pd.DateOffset(months=3) - pd.Timedelta(days=1)
        dates = pd.date_range(start=ref_start, end=ref_end)

        min_len = min(
            min(len(p) for p in ensemble_preds),
            min(len(s) for s in sim_3_months)
        )
        dates = dates[:min_len]

        all_preds = np.vstack([p[:min_len] for p in ensemble_preds])
        median_pred = np.nanmedian(all_preds, axis=0)
        percentile_10 = np.nanpercentile(all_preds, 10, axis=0)
        percentile_90 = np.nanpercentile(all_preds, 90, axis=0)

        all_sim = np.vstack([s[:min_len] for s in sim_3_months])
        median_sim = np.nanmedian(all_sim, axis=0)
        percentile_10_sim = np.nanpercentile(all_sim, 10, axis=0)
        percentile_90_sim = np.nanpercentile(all_sim, 90, axis=0)
        # percentile_0_sim = np.nanpercentile(all_sim, 0, axis=0)
        # percentile_100_sim = np.nanpercentile(all_sim, 100, axis=0)

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.fill_between(dates, percentile_10_sim, percentile_90_sim, color="blue", alpha=0.2, label="10th-90th Percentile (Sim)")
        # ax.fill_between(dates, percentile_0_sim, percentile_100_sim, color="blue", alpha=0.1, label="Full distribution (Sim)")
        ax.fill_between(dates, percentile_10, percentile_90, color="red", alpha=0.2, label="10th-90th Percentile (ESP)")
        ax.plot(dates, median_sim, color="blue", label="Median Simulated", linewidth=2)
        ax.plot(dates, median_pred, color="red", label="Median ESP Prediction", linewidth=2)
        ax.set_xlabel("Date")
        ax.set_ylabel("Streamflow (mm/d)")
        ax.set_title(f"ESP vs Simulated - Basin: {basin} - Month: {month}")
        ax.legend()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
        fig.savefig(esp_hydrographs_output_dir / f"{basin}_month_{month}_hydrograph.png", dpi=300)
        plt.close(fig)

        print(f"Plotted month {month} for basin {basin}")

Plotted month 1 for basin CAMELS_UY_2
Plotted month 2 for basin CAMELS_UY_2
Plotted month 3 for basin CAMELS_UY_2
Plotted month 4 for basin CAMELS_UY_2
Plotted month 5 for basin CAMELS_UY_2
Plotted month 6 for basin CAMELS_UY_2
Plotted month 7 for basin CAMELS_UY_2
Plotted month 8 for basin CAMELS_UY_2
Plotted month 9 for basin CAMELS_UY_2
Plotted month 10 for basin CAMELS_UY_2
Plotted month 11 for basin CAMELS_UY_2
Plotted month 12 for basin CAMELS_UY_2
Plotted month 1 for basin CAMELS_UY_3
Plotted month 2 for basin CAMELS_UY_3
Plotted month 3 for basin CAMELS_UY_3
Plotted month 4 for basin CAMELS_UY_3
Plotted month 5 for basin CAMELS_UY_3
Plotted month 6 for basin CAMELS_UY_3
Plotted month 7 for basin CAMELS_UY_3
Plotted month 8 for basin CAMELS_UY_3
Plotted month 9 for basin CAMELS_UY_3
Plotted month 10 for basin CAMELS_UY_3
Plotted month 11 for basin CAMELS_UY_3
Plotted month 12 for basin CAMELS_UY_3
Plotted month 1 for basin CAMELS_UY_5
Plotted month 2 for basin CAMELS_UY_5
Plotte